Function: create fixed windows for downstream feature extraction.

Windowing strategy:
1. Load filtered dataset from data/interim/filtered/filtered_dataset.csv (fallback: cleaned dataset),
2. Detect continuous segments within each run using a gap threshold,
3. Create fixed-size sliding windows per segment (2 seconds at 100 Hz = 200 samples),
4. Assign each window a majority label,
5. Save window metadata to data/processed/windowed_metadata.csv.

This prevents windows from crossing gaps created by cleaning and avoids boundary artifacts.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.windowing import (
    detect_continuous_segments,
    create_windows,
    save_windowed_run,
)

In [2]:
# Load filtered dataset (preferred) with fallback to cleaned dataset
filtered_data_path = PROJECT_ROOT / "data" / "interim" / "filtered" / "filtered_dataset.csv"
cleaned_data_path = PROJECT_ROOT / "data" / "interim" / "cleaned" / "cleaned_dataset.csv"

if filtered_data_path.exists():
    data_path = filtered_data_path
    print("Using filtered dataset from 04b_filtering.ipynb")
elif cleaned_data_path.exists():
    data_path = cleaned_data_path
    print("Filtered dataset not found; falling back to cleaned dataset")
else:
    raise FileNotFoundError(
        f"Neither filtered nor cleaned dataset was found:\n"
        f"- {filtered_data_path}\n"
        f"- {cleaned_data_path}\n"
        "Please run 04_cleaning.ipynb (and preferably 04b_filtering.ipynb) first."
    )

df = pd.read_csv(data_path, low_memory=False)

print(f"Loaded dataset: {data_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Unique run_ids: {df['run_id'].nunique()}")
print("\nPer-run sample counts:")
print(df.groupby('run_id').size().sort_index().to_string())

Using filtered dataset from 04b_filtering.ipynb
Loaded dataset: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/filtered/filtered_dataset.csv
Shape: (303488, 14)
Columns: ['t', 't_rel', 'run_id', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'v1', 'v2', 'label', 'net_speed', 'yaw_rate']
Unique run_ids: 4

Per-run sample counts:
run_id
log_20260223_142511.490     47926
log_20260226_102148.990     16986
log_20260309_141435.414    160762
log_20260326_120021.508     77814


In [3]:
# Segment-aware fixed-length windowing
SAMPLE_RATE = 100  # Hz
WINDOW_SECONDS = 2.0
OVERLAP_RATIO = 0.5  # 50% overlap
GAP_THRESHOLD = 0.025

window_size = int(WINDOW_SECONDS * SAMPLE_RATE)
step_size = max(1, int(window_size * (1.0 - OVERLAP_RATIO)))
STEP_SECONDS = step_size / SAMPLE_RATE

window_feature_cols = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
missing_feature_cols = [c for c in window_feature_cols if c not in df.columns]
if missing_feature_cols:
    raise KeyError(f"Missing required IMU columns for windowing: {missing_feature_cols}")

required_cols = ['run_id', 't_rel', 'label']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise KeyError(f"Missing required columns for windowing: {missing_required}")

segments = detect_continuous_segments(df, run_col='run_id', tcol='t_rel', gap_threshold=GAP_THRESHOLD)
segment_df = pd.DataFrame(segments)[['run_id', 'segment_id', 'n_samples', 'start_time', 'end_time']].copy()
segment_df['duration_s'] = segment_df['end_time'] - segment_df['start_time']
segment_df = segment_df.sort_values(['run_id', 'segment_id']).reset_index(drop=True)

print("Windowing configuration:")
print(f"  SAMPLE_RATE: {SAMPLE_RATE} Hz")
print(f"  WINDOW_SECONDS: {WINDOW_SECONDS}s ({window_size} samples)")
print(f"  OVERLAP_RATIO: {OVERLAP_RATIO:.0%}")
print(f"  STEP_SECONDS: {STEP_SECONDS}s ({step_size} samples)")
print(f"  GAP_THRESHOLD: {GAP_THRESHOLD}s")
print(f"  Window channels: {window_feature_cols}")
print()

print(f"Detected segments: {len(segment_df)}")
print(segment_df.groupby('run_id')['segment_id'].count().rename('n_segments').to_string())

windows_df = create_windows(
    df=df,
    feature_cols=window_feature_cols,
    window_size=window_size,
    step_size=step_size,
    label_col='label',
    run_col='run_id',
    tcol='t_rel',
    gap_threshold=GAP_THRESHOLD,
    min_samples=window_size,
    label_strategy='majority',
    include_series=False,
)

if windows_df.empty:
    raise RuntimeError("No windows were created. Try reducing WINDOW_SECONDS or STEP_SECONDS.")

windows_df['window_id'] = range(len(windows_df))
windows_df = windows_df[
    ['window_id', 'run_id', 'segment_id', 'window_start_idx', 'window_end_idx', 't_start', 't_end', 'n_samples', 'label']
].copy()

print(f"\nCreated windows: {len(windows_df)}")
print("Label distribution:")
print(windows_df['label'].value_counts().sort_index().to_string())
print("\nWindows per run:")
print(windows_df.groupby('run_id').size().sort_index().to_string())

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
window_path = output_dir / "windowed_metadata.csv"
save_windowed_run(windows_df, window_path)

print(f"\nSaved window metadata to: {window_path}")
print(windows_df.head())

Windowing configuration:
  SAMPLE_RATE: 100 Hz
  WINDOW_SECONDS: 2.0s (200 samples)
  OVERLAP_RATIO: 50%
  STEP_SECONDS: 1.0s (100 samples)
  GAP_THRESHOLD: 0.025s
  Window channels: ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

Detected segments: 1265
run_id
log_20260223_142511.490    191
log_20260226_102148.990     92
log_20260309_141435.414    781
log_20260326_120021.508    201

Created windows: 2534
Label distribution:
label
dry_dirt_track      652
footpath            543
grass               488
muddy_dirt_track    145
smooth_terrain      706

Windows per run:
run_id
log_20260223_142511.490     391
log_20260226_102148.990     145
log_20260309_141435.414    1310
log_20260326_120021.508     688

Saved window metadata to: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/processed/windowed_metadata.csv
   window_id                   run_id  segment_id  window_start_idx  \
0          0  log_20260223_142511.490           7                 0   
1          1  l